# SPRO Multi-Turn Jailbreak Training (Clean Import Version)

This notebook imports the SPRO module from `src/spro/` for cleaner iteration.

**Key Features:**
- All core code in `src/spro/` module
- This notebook: config + model loading + training loop
- Easy to iterate on individual components

**Module Structure:**
```
src/spro/
  __init__.py        # Exports all components
  config.py          # SPROConfig dataclass
  data_structures.py # AttackPlan, ExecutedEpisode
  parsing.py         # parse_sema_output
  intent_tracker.py  # IntentTracker (cosine similarity)
  rewards.py         # Reward functions
  prompts.py         # SEMA_SYSTEM_PROMPT, JUDGE_PROMPT
  api_client.py      # OpenRouterClient
  spro_core.py       # compute_log_prob_ratios, compute_spro_advantages
  trainer.py         # SingleExampleSPROTrainer
```

## 1. Setup - Clone Repo & Install Dependencies

In [ ]:
import os

# Clone repo if not already present
if not os.path.exists('JB_mech'):
    !git clone https://github.com/ChuloIva/JB_mech.git
    print("Repo cloned!")
else:
    # Pull latest changes
    !cd JB_mech && git pull
    print("Repo already exists, pulled latest.")

In [ ]:
%%capture
# Install dependencies
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    import torch; import re; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets>=4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install sentence-transformers openai
!pip install liger-kernel vllm
!pip install nest_asyncio tenacity httpx aiohttp

In [ ]:
# Add src to path for imports
import sys
sys.path.insert(0, 'JB_mech/src')

print("Installation complete!")

## 2. Configuration

In [ ]:
from spro import SPROConfig

# Override defaults as needed
config = SPROConfig(
    # Model
    model_name="Koalacrown/qwen3-4b-multiturn-sft-16bit",
    lora_rank=32,
    
    # Training
    learning_rate=1e-5,
    group_size=8,
    max_attempts=200,
    
    # SPRO advantages
    msa_scale=0.1,
    advantage_clip=5.0,
    normalize_advantages=True,
    use_gae_advantages=False,  # Set True to use V2
    
    # Rewards - IDA (Intent-Drift-Aware) scoring from SEMA paper
    # R_IDA = 0.5 * Alignment * (Risk + Detail) / 100
    # Alignment: How well response addresses original intent (0-10)
    # Risk: Compliance risk level of response (0-10)
    # Detail: How thorough/specific the response is (0-10)
    outcome_weight=0.85,      # Weight for IDA reward
    divergence_weight=0.15,   # Bonus for indirect successful attacks
    
    # API
    openrouter_api_key="",  # Set your key
    target_model="meta-llama/llama-3.1-8b-instruct",
    judge_model="google/gemma-3-27b-it",
    
    # Checkpointing
    checkpoint_dir="./spro_checkpoints",
)

print(f"Config loaded:")
print(f"  Model: {config.model_name}")
print(f"  Group size: {config.group_size}")
print(f"  Reward: IDA scoring (Alignment × (Risk + Detail))")
print(f"  Weights: outcome={config.outcome_weight}, divergence={config.divergence_weight}")
print(f"  SPRO: msa_scale={config.msa_scale}, advantage_clip={config.advantage_clip}")

## 3. HuggingFace Login & Setup

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os

# Mount Google Drive (Colab) or create local checkpoint dir
try:
    from google.colab import drive
    drive.mount('/content/drive')
    config.checkpoint_dir = "/content/drive/MyDrive/mt_rl_checkpoints/spro"
    print("Google Drive mounted.")
except ImportError:
    print("Not in Colab, using local checkpoint directory.")

os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(f"{config.checkpoint_dir}/rollouts", exist_ok=True)
print(f"Checkpoints: {config.checkpoint_dir}")

## 4. Load Models

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load policy model with vLLM
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.model_name,
    max_seq_length=config.max_seq_length,
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=config.lora_rank,
    gpu_memory_utilization=config.gpu_memory_utilization,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=config.lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(f"Policy model loaded: {config.model_name}")

In [ ]:
# Load reference model (frozen)
ref_model, _ = FastLanguageModel.from_pretrained(
    model_name=config.model_name,
    max_seq_length=config.max_seq_length,
    load_in_4bit=False,
    fast_inference=False,
)

for param in ref_model.parameters():
    param.requires_grad = False

print("Reference model loaded and frozen.")

## 5. Setup Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3-thinking")

# Fix template to ensure <think> tag
test_msgs = [{"role": "user", "content": "test"}]
test_output = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)

if "<think>" not in test_output:
    print("[!] Applying <think> tag fix...")
    tokenizer.chat_template = '''{% for message in messages %}{% if message['role'] == 'system' %}<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}<|im_start|>assistant
{{ message['content'] }}<|im_end|>
{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant
<think>
{% endif %}'''

test_output = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print(f"Template contains <think>: {'<think>' in test_output}")

## 6. Load Dataset

In [ ]:
from datasets import load_dataset

# Load OBLITERATUS dataset
obliteratus = load_dataset("Koalacrown/obliteratus-jailbreak-prompts", split="train")
print(f"Loaded {len(obliteratus)} total examples")

# Filter by tier
tier1_data = obliteratus.filter(lambda x: x["tier"] == "Tier 1: Standard")
tier2_data = obliteratus.filter(lambda x: x["tier"] == "Tier 2: Elevated")
tier3_data = obliteratus.filter(lambda x: x["tier"] == "Tier 3: Critical")

print(f"Tier 1 (easy): {len(tier1_data)}")
print(f"Tier 2 (medium): {len(tier2_data)}")
print(f"Tier 3 (hard): {len(tier3_data)}")

# Train on Tier 2
training_intents = [ex["harmful"] for ex in tier2_data]
print(f"\nTraining on {len(training_intents)} Tier 2 intents")

## 7. Initialize Trainer

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from spro import IntentTracker, SingleExampleSPROTrainer

# Initialize components
intent_tracker = IntentTracker()

trainer = SingleExampleSPROTrainer(
    policy_model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    intent_tracker=intent_tracker,
    config=config,
)

print("Trainer initialized.")
print(f"Training on {len(training_intents)} intents")
print(f"Group size: {config.group_size}")

## 8. Test API Connection

In [ ]:
import asyncio

if config.openrouter_api_key:
    result = asyncio.get_event_loop().run_until_complete(
        trainer.api_client.test_connection()
    )
    print(f"API test: {result[:50]}...")
else:
    print("[!] Set config.openrouter_api_key to test API")

## 9. Training

In [ ]:
# Single intent test
# test_intent = training_intents[0]
# result = asyncio.get_event_loop().run_until_complete(trainer.train_on_intent(test_intent))
# print(f"Result: {result}")

In [ ]:
from tqdm.auto import tqdm

async def train_all_intents():
    """Train on all intents sequentially."""
    results = []
    
    for i, intent in enumerate(tqdm(training_intents, desc="Training intents")):
        print(f"\n{'='*60}")
        print(f"Intent {i+1}/{len(training_intents)}")
        print(f"{'='*60}")
        
        result = await trainer.train_on_intent(intent)
        results.append(result)
        trainer.training_stats.append(result)
        
        # Save checkpoint periodically
        if (i + 1) % config.save_every_n_intents == 0:
            trainer.save_checkpoint(f"{config.checkpoint_dir}/checkpoint_{i+1}")
    
    return results

# Run training (uncomment)
# results = asyncio.get_event_loop().run_until_complete(train_all_intents())

In [ ]:
# Quick test with a few intents
async def train_quick_test(n_intents: int = 3):
    results = []
    for i, intent in enumerate(training_intents[:n_intents]):
        print(f"\nIntent {i+1}/{n_intents}")
        result = await trainer.train_on_intent(intent)
        results.append(result)
        trainer.training_stats.append(result)
    return results

# Run quick test (uncomment)
# results = asyncio.get_event_loop().run_until_complete(train_quick_test(1))

## 10. Analysis

In [ ]:
def analyze_results(trainer):
    """Analyze training results."""
    stats = trainer.training_stats
    
    if not stats:
        print("No training stats yet.")
        return
    
    successes = sum(1 for s in stats if s["success"])
    total = len(stats)
    
    print(f"\n{'='*60}")
    print("TRAINING SUMMARY")
    print(f"{'='*60}")
    print(f"Intents trained: {total}")
    print(f"Successes: {successes} ({100*successes/total:.1f}%)")
    
    attempts = [s["attempts"] for s in stats]
    print(f"\nAttempts per intent: min={min(attempts)}, max={max(attempts)}, mean={sum(attempts)/len(attempts):.1f}")
    
    scores = [s["best_score"] for s in stats]
    print(f"\nBest score distribution:")
    for score in [1, 2, 3, 4]:
        count = scores.count(score)
        print(f"  Score {score}: {count}")

# analyze_results(trainer)

## 11. Save Checkpoint

In [ ]:
# trainer.save_checkpoint(f"{config.checkpoint_dir}/final")

In [ ]:
# Push to HuggingFace
# HF_USERNAME = "Koalacrown"
# model.push_to_hub(f"{HF_USERNAME}/qwen3-4b-spro-multiturn-lora")
# tokenizer.push_to_hub(f"{HF_USERNAME}/qwen3-4b-spro-multiturn-lora")